In [ ]:
!pip install -q langchain langchain-community langchain-huggingface pypdf chromadb sentence-transformers gradio langchain-openai langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/13

In [ ]:
import uuid
import os
from typing import List
import gradio as gr

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


In [ ]:
class RAGApplication:
    def __init__(self):
        self.session_id = str(uuid.uuid4())
        self.vectorstore = None
        self.qa_chain = None
        self.embeddings = None
        self.docs_loaded = False

        print(f"✓ RAG Application initialized")
        print(f"  Session ID: {self.session_id}")

    def load_pdf(self, pdf_path: str):
        try:
            doc_id = str(uuid.uuid4())

            loader = PyPDFLoader(pdf_path)
            documents = loader.load()


            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=1000,
                chunk_overlap=200,
                length_function=len
            )
            chunks = text_splitter.split_documents(documents)


            for chunk in chunks:
                chunk.metadata['doc_id'] = doc_id
                chunk.metadata['session_id'] = self.session_id

            return chunks, doc_id

        except Exception as e:
            raise Exception(f"Error loading PDF: {str(e)}")

    def create_vectorstore(self, chunks):

        try:

            self.embeddings = HuggingFaceEmbeddings(
                model_name="sentence-transformers/all-MiniLM-L6-v2"
            )

            # Create vector store
            self.vectorstore = Chroma.from_documents(
                documents=chunks,
                embedding=self.embeddings,
                collection_name=f"rag_collection_{self.session_id[:8]}"
            )

            self.docs_loaded = True
            print(f"✓ Vector store created with {len(chunks)} chunks")

        except Exception as e:
            raise Exception(f"Error creating vector store: {str(e)}")

    def setup_qa_chain(self, api_key: str = None, model_name: str = "meta-llama/llama-3.1-8b-instruct:free"):
        try:
            if not self.docs_loaded:
                raise Exception("No documents loaded. Please upload a PDF first.")

            # Setup retriever
            retriever = self.vectorstore.as_retriever(
                search_kwargs={"k": 3}
            )

            # Create a prompt template for RAG
            prompt = ChatPromptTemplate.from_template(
                """You are a helpful AI assistant analyzing a document. The context below contains excerpts from the document.

When users ask about "the pdf", "the document", "this file", or similar terms, they are asking about the CONTENT and information contained within it, not about the file itself.

Answer the user's question based on the context provided. Be conversational and helpful. If the question is vague (like "tell me about this" or "what's this about"), provide a comprehensive summary of the main topics, key information, and important details found in the context.

If you cannot find relevant information in the context to answer the question, say so politely.

Context from the document:
{context}

User's question: {input}

Your answer:"""
            )

            llm = None
            if api_key:
                models_to_try = [
                    model_name,
                    "meta-llama/llama-3.2-3b-instruct:free",
                    "qwen/qwen-2.5-7b-instruct",
                ]

                seen = set()
                models_to_try = [x for x in models_to_try if not (x in seen or seen.add(x))]

                for model in models_to_try:
                    try:
                        print(f"  → Trying OpenRouter with {model}...")
                        llm = ChatOpenAI(
                            api_key=api_key,
                            model=model,
                            base_url="https://openrouter.ai/api/v1",
                            temperature=0.3,
                            max_tokens=5000,
                        )



                        print(f"  → Testing model...")
                        test_result = llm.invoke("Hello")

                        print(f"  ✓ Successfully initialized and tested {model}")
                        break

                    except Exception as error:
                        error_msg = str(error)
                        print(f"  ✗ {model} failed: {error_msg[:100]}")
                        llm = None


                        if model != models_to_try[-1]:
                            print(f"  → Trying next model...")
                            continue

                if not llm:
                    print("  ⚠ All models failed. Falling back to retrieval-only mode.")
            else:
                print("⚠ No API key provided. Using retrieval-only mode.")

            if llm:
                question_answer_chain = create_stuff_documents_chain(llm, prompt)
                self.qa_chain = create_retrieval_chain(retriever, question_answer_chain)
                print("✓ QA chain configured with AI")
            else:
                self.qa_chain = None
                print("✓ QA chain configured (retrieval-only mode)")

        except Exception as e:
            raise Exception(f"Error setting up QA chain: {str(e)}")

    def query(self, question: str):
        """Query the RAG system."""
        try:
            if not self.docs_loaded:
                return "❌ No documents loaded. Please upload a PDF first.", ""


            query_id = str(uuid.uuid4())

            if self.qa_chain:

                print("  → Invoking QA chain...")


                result = self.qa_chain.invoke({"input": question})
                print(f"  → Result type: {type(result)}")
                print(f"  → Result keys: {result.keys() if isinstance(result, dict) else 'Not a dict'}")

                answer = result.get('answer', '')
                print(f"  → Raw answer: {repr(answer[:200])}")
                print(f"  → Answer length: {len(answer) if answer else 0}")


                if not answer or len(answer.strip()) < 10:
                    print("  ⚠ Model returned empty/short answer - this model may not be suitable for RAG")
                    answer = "⚠ The AI model returned an empty response. This might mean:\n" \
                            "1. The model is not suitable for text generation\n" \
                            "2. The question needs to be more specific\n" \
                            "3. Try selecting a different model like 'google/gemini-flash-1.5'\n\n" \
                            "Here's what the document contains instead:\n\n"


                    sources = result.get('context', [])
                    if sources:
                        for i, doc in enumerate(sources[:2], 1):
                            answer += f"\n📄 Excerpt {i}:\n{doc.page_content.strip()[:500]}...\n"
                    else:
                        answer += "No relevant content found."

                sources = result.get('context', [])
            else:

                print("  → Using retrieval-only mode...")
                retriever = self.vectorstore.as_retriever(search_kwargs={"k": 5})
                sources = retriever.invoke(question)

                if not sources or len(sources) == 0:
                    answer = "No relevant information found in the document for your question."
                else:

                    answer_parts = ["Here's what I found in the document:\n"]
                    for i, doc in enumerate(sources[:3], 1):
                        content = doc.page_content.strip()
                        answer_parts.append(f"\n📄 Excerpt {i}:\n{content}")
                    answer = "\n".join(answer_parts)
                    print(f"  ✓ Retrieved {len(sources)} relevant chunks")


            metadata = f"Query ID: {query_id}\nSession ID: {self.session_id}"

            return answer, metadata

        except Exception as e:
            import traceback
            error_details = f"{type(e).__name__}: {str(e)}"
            if not str(e):
                error_details = f"{type(e).__name__} (empty message)"
            print(f"  ❌ Exception in query: {error_details}")
            traceback.print_exc()
            return f"❌ Error: {error_details}", ""


rag_app = RAGApplication()

✓ RAG Application initialized
  Session ID: f0739da1-3e0a-4fdb-bd34-c69a8f0b1799


In [ ]:
def upload_pdf(pdf_file, api_key, model_name):
    """Handle PDF upload and processing."""
    try:
        if pdf_file is None:
            return "❌ Please upload a PDF file.", ""


        chunks, doc_id = rag_app.load_pdf(pdf_file.name)

        rag_app.create_vectorstore(chunks)


        rag_app.setup_qa_chain(api_key if api_key else None, model_name=model_name)

        message = f"✅ PDF loaded successfully!\n\nDocument ID: {doc_id}\nChunks created: {len(chunks)}"
        return message, ""

    except Exception as e:
        return f"❌ Error: {str(e)}", ""

def ask_question(question, chat_history):
    """Handle question answering."""
    if not question.strip():
        return chat_history, ""


    answer, metadata = rag_app.query(question)


    chat_history.append({"role": "user", "content": question})
    chat_history.append({"role": "assistant", "content": answer})

    return chat_history, metadata


with gr.Blocks(title="RAG Application with UID") as demo:
    gr.Markdown("# 📚 RAG Application with PDF Support")
    gr.Markdown("Upload a PDF document and ask questions about its content. Each query is tracked with unique IDs.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1️⃣ Upload PDF")
            pdf_input = gr.File(
                label="Select PDF File",
                file_types=[".pdf"]
            )
            model_input = gr.Dropdown(
                choices=[
                    "stepfun/step-3.5-flash:free",
                    "meta-llama/llama-3.2-3b-instruct:free",
                    "qwen/qwen-2.5-7b-instruct",
                ],
                value="stepfun/step-3.5-flash:free",
                label="AI Model",
                info="Free models end with :free"
            )
            api_key_input = gr.Textbox(
                label="OpenRouter API Key",
                placeholder="sk-or-v1-...",
                type="password",
                info="Get free key from https://openrouter.ai/keys"
            )
            upload_btn = gr.Button("📤 Upload & Process PDF", variant="primary")
            upload_status = gr.Textbox(
                label="Upload Status",
                lines=4,
                interactive=False
            )

        with gr.Column(scale=2):
            gr.Markdown("### 2️⃣ Ask Questions")
            chatbot = gr.Chatbot(
                label="Conversation",
                height=400,
                type="messages",
                render_markdown=True
            )
            question_input = gr.Textbox(
                label="Your Question",
                placeholder="Ask anything about the PDF... (Press Enter to submit)",
                lines=1
            )
            submit_btn = gr.Button("🚀 Ask", variant="primary")
            query_metadata = gr.Textbox(
                label="Query Metadata",
                lines=2,
                interactive=False
            )

            with gr.Row():
                clear_btn = gr.Button("🗑️ Clear Chat")


    upload_btn.click(
        fn=upload_pdf,
        inputs=[pdf_input, api_key_input, model_input],
        outputs=[upload_status, query_metadata]
    )

    submit_btn.click(
        fn=ask_question,
        inputs=[question_input, chatbot],
        outputs=[chatbot, query_metadata]
    ).then(
        lambda: "",
        outputs=question_input
    )

    question_input.submit(
        fn=ask_question,
        inputs=[question_input, chatbot],
        outputs=[chatbot, query_metadata]
    ).then(
        lambda: "",
        outputs=question_input
    )

    clear_btn.click(
        lambda: ([], ""),
        outputs=[chatbot, query_metadata]
    )

    gr.Markdown("""
    ### 📋 Instructions:
    1. Get a FREE OpenRouter API key from: https://openrouter.ai/keys
    2. Select an AI model (models ending with :free are completely free!)
    3. Enter your OpenRouter API key
    4. Upload a PDF file
    5. Click "Upload & Process PDF" to process the document
    6. Ask questions naturally - the AI understands casual questions!

    **Popular Models:**
    - **Step 3.5 Flash** (free): Fast and reliable for RAG
    - **Gemini Flash 1.5**: Fast Google model
    - **Llama 3.1 8B** (free): Great for general Q&A
    - **Claude 3.5 Sonnet**: Best quality (paid)
    - **GPT-4o Mini**: Balanced performance (paid)

    **Example Questions:**
    - "Tell me about the pdf"
    - "What's this document about?"
    - "Summarize the main points"
    - "What are the key details?"

    Visit https://openrouter.ai/models to see all available models and pricing.

    **Note:** Each query is tracked with a unique Query ID for your reference.
    """)

print("✓ Gradio interface created successfully!")

✓ Gradio interface created successfully!


/tmp/ipython-input-844534382.py:72: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


In [ ]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c8c9335f66750aedab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1134, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Vector store created with 59 chunks
  → Trying OpenRouter with stepfun/step-3.5-flash:free...
  → Testing model...
  ✓ Successfully initialized and tested stepfun/step-3.5-flash:free
✓ QA chain configured with AI
  → Invoking QA chain...
  → Result type: <class 'dict'>
  → Result keys: dict_keys(['input', 'context', 'answer'])
  → Raw answer: 'Based on the provided excerpts, this document appears to be a section from an academic or review article, likely from a journal like the *International Journal of Medical Mushrooms* or a publication o'
  → Answer length: 1895
